In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_table = "retailnova.bronze.orders"
silver_table = "retailnova.silver.orders"

In [0]:
df = spark.table(bronze_table)

print("Bronze records:", df.count())


Bronze records: 1003000


In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
# Remove records without important IDs
df = df.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("product_id").isNotNull() &
    col("store_id").isNotNull()
)

In [0]:
# Trim string columns
df = df.withColumn("order_id", trim(col("order_id")))
df = df.withColumn("customer_id", trim(col("customer_id")))
df = df.withColumn("product_id", trim(col("product_id")))
df = df.withColumn("store_id", trim(col("store_id")))
df = df.withColumn("status", trim(col("status")))
df = df.withColumn("payment_method", trim(col("payment_method")))



In [0]:
# Ensure numeric columns
df = df.withColumn("quantity", col("quantity").cast("double"))
df = df.withColumn("unit_price", col("unit_price").cast("double"))
df = df.withColumn("discount_amount", col("discount_amount").cast("double"))

In [0]:
# Quality status
df = df.withColumn(
    "quality_status",
    when(
        col("quantity").isNull() |
        col("unit_price").isNull() |
        (col("quantity") <= 0) |
        (col("unit_price") < 0) |
        (col("discount_amount") < 0),
        "INVALID"
    ).otherwise("VALID")
)

In [0]:
# Remove exact duplicate orders
df = df.dropDuplicates()

In [0]:
print("Silver records:", df.count())

Silver records: 999350


In [0]:




display(df)

order_id,customer_id,product_id,store_id,order_timestamp,quantity,unit_price,discount_amount,status,payment_method,updated_at,_rescued_data,quality_status
O0551695,C047990,P009910,ST0243,2026-05-22T07:46:03.000Z,4.0,693.58,688.32,Completed,Net Banking,2026-05-22T07:46:03.000Z,null,VALID
O0551702,C034705,P007738,ST0413,2026-03-10T09:04:48.000Z,4.0,818.31,81.74,Completed,Credit Card,2026-03-11T01:04:48.000Z,null,VALID
O0551706,C038387,P005684,ST0102,2026-02-19T16:08:36.000Z,3.0,279.78,76.14,Completed,UPI,2026-02-21T10:08:36.000Z,null,VALID
O0551807,C031381,P001061,ST0027,2026-08-06T17:27:15.000Z,5.0,1643.88,931.15,Completed,UPI,2026-08-07T06:27:15.000Z,null,VALID
O0551808,C072053,P000122,ST0061,2026-07-14T08:56:22.000Z,5.0,1537.11,1051.96,Completed,Credit Card,2026-07-15T06:56:22.000Z,null,VALID
O0551857,C035101,P005430,ST0445,2026-02-18T22:26:31.000Z,2.0,321.08,81.69,Completed,Net Banking,2026-02-19T15:26:31.000Z,null,VALID
O0551866,C036083,P007687,ST0392,2026-04-26T15:21:56.000Z,2.0,1366.7,614.1,Shipped,Cash,2026-04-28T02:21:56.000Z,null,VALID
O0551939,C084423,P003473,ST0117,2026-04-18T08:21:44.000Z,1.0,3800.99,335.34,Completed,UPI,2026-04-19T15:21:44.000Z,null,VALID
O0552013,C066146,P001601,ST0117,2026-07-23T22:50:18.000Z,4.0,3637.63,2817.83,Completed,UPI,2026-07-26T21:50:18.000Z,null,VALID
O0552041,C021185,P009252,ST0013,2026-08-16T06:35:51.000Z,3.0,1511.88,117.7,Completed,UPI,2026-08-16T09:35:51.000Z,null,VALID


In [0]:
# Quality check
df.groupBy("quality_status").count().show()

+--------------+------+
|quality_status| count|
+--------------+------+
|       INVALID|   849|
|         VALID|998501|
+--------------+------+



In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print("Orders Silver created successfully")


Orders Silver created successfully
